In [1]:
import os

COTTON_DATASET_PATH = r"D:\AgriMithra datasets\Cotton Leaf Image Dataset for Disease Classificati\Cotton Leaf Image Dataset for Disease Classificati\Cotton_Original_Dataset\Cotton_Original_Dataset"

print("Dataset exists:", os.path.exists(COTTON_DATASET_PATH))

if os.path.exists(COTTON_DATASET_PATH):
    print("\nCotton classes:")
    for folder in sorted(os.listdir(COTTON_DATASET_PATH)):
        print(folder)

Dataset exists: True

Cotton classes:
Alternaria Leaf Spot
Bacterial Blight
Fusarium Wilt
Healthy Leaf
Verticillium Wilt


In [2]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

cotton_train_ds = tf.keras.utils.image_dataset_from_directory(
    COTTON_DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

cotton_val_ds = tf.keras.utils.image_dataset_from_directory(
    COTTON_DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# IMPORTANT: get class names BEFORE prefetch()
cotton_class_names = cotton_train_ds.class_names

print("\nCotton classes:", cotton_class_names)
print("Number of classes:", len(cotton_class_names))

Found 1373 files belonging to 5 classes.
Using 1099 files for training.
Found 1373 files belonging to 5 classes.
Using 274 files for validation.

Cotton classes: ['Alternaria Leaf Spot', 'Bacterial Blight', 'Fusarium Wilt', 'Healthy Leaf', 'Verticillium Wilt']
Number of classes: 5


In [3]:
AUTOTUNE = tf.data.AUTOTUNE

cotton_train_ds = cotton_train_ds.prefetch(
    buffer_size=AUTOTUNE
)

cotton_val_ds = cotton_val_ds.prefetch(
    buffer_size=AUTOTUNE
)

print("Cotton dataset prefetch completed!")
print("Classes:", cotton_class_names)

Cotton dataset prefetch completed!
Classes: ['Alternaria Leaf Spot', 'Bacterial Blight', 'Fusarium Wilt', 'Healthy Leaf', 'Verticillium Wilt']


In [4]:
import os

cotton_class_names = sorted([
    folder for folder in os.listdir(COTTON_DATASET_PATH)
    if os.path.isdir(os.path.join(COTTON_DATASET_PATH, folder))
])

print("Cotton classes:")
for i, name in enumerate(cotton_class_names):
    print(i, name)

print("\nNumber of classes:", len(cotton_class_names))

Cotton classes:
0 Alternaria Leaf Spot
1 Bacterial Blight
2 Fusarium Wilt
3 Healthy Leaf
4 Verticillium Wilt

Number of classes: 5


In [5]:
from tensorflow.keras import layers, models

num_cotton_classes = len(cotton_class_names)

cotton_model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    # Normalize pixel values
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(num_cotton_classes, activation='softmax')
])

cotton_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,405 (431.27 KB)

 Trainable params: 110,405 (431.27 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
num_cotton_classes = len(cotton_class_names)

In [7]:
cotton_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cotton_history = cotton_model.fit(
    cotton_train_ds,
    validation_data=cotton_val_ds,
    epochs=5
)

Epoch 1/5


c:\Users\udata\anaconda3\envs\agrimitra-ml\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


69/69 ━━━━━━━━━━━━━━━━━━━━ 33s 436ms/step - accuracy: 0.3385 - loss: 1.4907 - val_accuracy: 0.5693 - val_loss: 1.1372
Epoch 2/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 40s 417ms/step - accuracy: 0.5396 - loss: 1.0693 - val_accuracy: 0.6022 - val_loss: 0.8810
Epoch 3/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 32s 458ms/step - accuracy: 0.7116 - loss: 0.7757 - val_accuracy: 0.8029 - val_loss: 0.5602
Epoch 4/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 36s 513ms/step - accuracy: 0.7798 - loss: 0.5755 - val_accuracy: 0.8394 - val_loss: 0.4083
Epoch 5/5
69/69 ━━━━━━━━━━━━━━━━━━━━ 39s 558ms/step - accuracy: 0.8071 - loss: 0.4695 - val_accuracy: 0.8358 - val_loss: 0.3968


In [8]:
import os

MODEL_DIR = r"C:\AgriMithra\AgriMitra\backend\models"

os.makedirs(MODEL_DIR, exist_ok=True)

COTTON_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "cotton_model.keras"
)

cotton_model.save(COTTON_MODEL_PATH)

print("Cotton model saved successfully!")
print("Saved at:", COTTON_MODEL_PATH)

Cotton model saved successfully!
Saved at: C:\AgriMithra\AgriMitra\backend\models\cotton_model.keras


In [9]:
print(os.path.exists(COTTON_MODEL_PATH))

True


In [21]:
import tensorflow as tf
import numpy as np
import os

# Choose another class
test_class = cotton_class_names[1]

TEST_FOLDER = os.path.join(
    COTTON_DATASET_PATH,
    test_class
)

# Get available images
images = [
    f for f in os.listdir(TEST_FOLDER)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

# Pick an image
IMAGE_PATH = os.path.join(TEST_FOLDER, images[0])

print("Testing image:", IMAGE_PATH)
print("Actual class:", test_class)

# Load image
img = tf.keras.utils.load_img(
    IMAGE_PATH,
    target_size=(224, 224)
)

img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

# Predict
predictions = cotton_model.predict(img_array, verbose=0)

predicted_index = np.argmax(predictions[0])
predicted_class = cotton_class_names[predicted_index]
confidence = predictions[0][predicted_index] * 100

print("\nPredicted class:", predicted_class)
print("Confidence:", round(float(confidence), 2), "%")

Testing image: D:\AgriMithra datasets\Cotton Leaf Image Dataset for Disease Classificati\Cotton Leaf Image Dataset for Disease Classificati\Cotton_Original_Dataset\Cotton_Original_Dataset\Bacterial Blight\bacterial_(1).png
Actual class: Bacterial Blight

Predicted class: Bacterial Blight
Confidence: 50.06 %


In [20]:
val_loss, val_accuracy = cotton_model.evaluate(
    cotton_val_ds,
    verbose=1
)

print("\nValidation Loss:", round(val_loss, 4))
print("Validation Accuracy:", round(val_accuracy * 100, 2), "%")

18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.8723 - loss: 0.3416

Validation Loss: 0.3416
Validation Accuracy: 87.23 %


In [10]:
import tensorflow as tf
import numpy as np
import os

correct = 0
total = 0

for actual_class in cotton_class_names:

    class_folder = os.path.join(
        COTTON_DATASET_PATH,
        actual_class
    )

    images = [
        f for f in os.listdir(class_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if len(images) == 0:
        continue

    # Take first image from this class
    image_path = os.path.join(class_folder, images[0])

    img = tf.keras.utils.load_img(
        image_path,
        target_size=(224, 224)
    )

    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = cotton_model.predict(
        img_array,
        verbose=0
    )

    predicted_index = np.argmax(predictions[0])

    predicted_class = cotton_class_names[predicted_index]

    confidence = np.max(predictions[0]) * 100

    total += 1

    if predicted_class == actual_class:
        correct += 1
        result = "CORRECT"
    else:
        result = "WRONG"

    print("--------------------------------")
    print("Actual     :", actual_class)
    print("Predicted  :", predicted_class)
    print("Confidence :", round(float(confidence), 2), "%")
    print("Result     :", result)


print("\n==============================")
print("Correct:", correct, "/", total)
print("Test Accuracy:", round((correct / total) * 100, 2), "%")

--------------------------------
Actual     : Alternaria Leaf Spot
Predicted  : Verticillium Wilt
Confidence : 57.57 %
Result     : WRONG
--------------------------------
Actual     : Bacterial Blight
Predicted  : Verticillium Wilt
Confidence : 62.24 %
Result     : WRONG
--------------------------------
Actual     : Fusarium Wilt
Predicted  : Fusarium Wilt
Confidence : 64.84 %
Result     : CORRECT
--------------------------------
Actual     : Healthy Leaf
Predicted  : Healthy Leaf
Confidence : 97.61 %
Result     : CORRECT
--------------------------------
Actual     : Verticillium Wilt
Predicted  : Verticillium Wilt
Confidence : 85.86 %
Result     : CORRECT

Correct: 3 / 5
Test Accuracy: 60.0 %


In [11]:
print(cotton_class_names)


['Alternaria Leaf Spot', 'Bacterial Blight', 'Fusarium Wilt', 'Healthy Leaf', 'Verticillium Wilt']
